In [ ]:
# Load libraries
library(MetaNeighbor)
library(Seurat)
library(SingleCellExperiment)
library(Matrix)
library(gplots)
library(RColorBrewer)

mainDir <- "/data/ebaird/scRNAseq/SCENTINELsep25/MetaNeighbor_analysis"
figDir <- paste0(mainDir, "/figures")

dir.create(mainDir, showWarnings = FALSE, recursive = TRUE)
dir.create(figDir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
seurat1 <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep24/QC_clustering/merged_clusters.rds")
seurat2 <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep25/QC_clustering/final_clusters.rds")

In [ ]:
Idents(seurat1) <- seurat1@meta.data$seurat_clusters
Idents(seurat2) <- seurat2@meta.data$seurat_clusters

# Rename cluster identities to avoid overlap
Idents(seurat1) <- paste0("S24_", Idents(seurat1))
Idents(seurat2) <- paste0("S25_", Idents(seurat2))

# New meta.data column for the new cluster identities
seurat1$cell_type <- Idents(seurat1)
seurat2$cell_type <- Idents(seurat2)

unique(seurat1@meta.data$cell_type)
unique(seurat2@meta.data$cell_type)

In [ ]:
# Alternative approach: Extract specific layer data

# Step 1: Add study identifiers
seurat1$study_id <- "study1"
seurat2$study_id <- "study2"

# Step 2: Get common genes
common_genes <- intersect(rownames(seurat1), rownames(seurat2))
seurat1 <- seurat1[common_genes, ]
seurat2 <- seurat2[common_genes, ]

# Step 3: Extract counts from specific layer (e.g., "counts" or "data")
counts1 <- LayerData(seurat1, assay = "RNA", layer = "counts")
counts2 <- LayerData(seurat2, assay = "RNA", layer = "counts")

# Step 4: Combine counts matrices
combined_counts <- cbind(counts1, counts2)

# Step 5: Get metadata
metadata_df <- rbind(
  data.frame(
    study_id = seurat1$study_id,
    cell_type = seurat1$cell_type,  # Adjust column name
    row.names = colnames(counts1)
  ),
  data.frame(
    study_id = seurat2$study_id,
    cell_type = seurat2$cell_type,  # Adjust column name
    row.names = colnames(counts2)
  )
)

# Step 6: Create SingleCellExperiment object
sce <- SingleCellExperiment(
  assays = list(counts = combined_counts),
  colData = metadata_df
)

# Optional: Add logcounts if available
if ("data" %in% Layers(seurat1)) {
  data1 <- LayerData(seurat1, assay = "RNA", layer = "data")
  data2 <- LayerData(seurat2, assay = "RNA", layer = "data")
  combined_data <- cbind(data1, data2)
  assay(sce, "logcounts") <- combined_data
}

In [ ]:
# Identify highly variable genes
var_genes <- variableGenes(dat = sce, exp_labels = sce$study_id)
print(paste("Number of variable genes:", length(var_genes)))

# Run unsupervised MetaNeighbor (use fast_version = TRUE for larger datasets)
celltype_NV <- MetaNeighborUS(
    var_genes = var_genes,
    dat = sce,
    study_id = sce$study_id,
    cell_type = sce$cell_type,
    fast_version = TRUE  # Use FALSE for smaller datasets (<10K cells)
)

# Plot heatmap of results
cols <- rev(colorRampPalette(RColorBrewer::brewer.pal(11, "RdYlBu"))(100))
breaks <- seq(0, 1, length = 101)

pdf(paste0(figDir, "MetaNeighbor_heatmap.pdf"), width = 10, height = 8)
heatmap.2(celltype_NV,
          margins = c(10, 10),
          keysize = 1,
          key.xlab = "AUROC",
          key.title = NULL,
          trace = "none",
          density.info = "none",
          col = cols,
          breaks = breaks,
          cexRow = 0.7,
          cexCol = 0.7,
          main = "Cell Type Similarity Across Datasets")
dev.off()

# Identify top hits (AUROC > 0.9 indicates strong matches)
top_hits <- topHits(
    cell_NV = celltype_NV,
    dat = sce,
    study_id = sce$study_id,
    cell_type = sce$cell_type,
    threshold = 0.9
)
print("Top matching cell types:")
print(top_hits)

In [ ]:
# Install bioconductor packages for Drosophila annotations
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install(c("org.Dm.eg.db", "GO.db", "AnnotationDbi"))

library(org.Dm.eg.db)
library(GO.db)
library(AnnotationDbi)

# Create GO gene sets for Drosophila
create_drosophila_genesets <- function(gene_universe = NULL, min_genes = 10, max_genes = 1000) {
    # Get all GO terms for Drosophila
    go_terms <- keys(org.Dm.eg.db, keytype = "GO")
    
    # Get genes for each GO term
    go2genes <- mapIds(org.Dm.eg.db,
                       keys = go_terms,
                       column = "SYMBOL",
                       keytype = "GO",
                       multiVals = "list")
    
    # Filter by size
    gene_set_sizes <- sapply(go2genes, function(x) length(na.omit(x)))
    valid_terms <- names(go2genes)[gene_set_sizes >= min_genes & gene_set_sizes <= max_genes]
    
    # Filter gene sets
    GOdrosophila <- go2genes[valid_terms]
    
    # Get GO term descriptions
    term_names <- select(GO.db, keys = valid_terms, columns = "TERM")$TERM
    names(GOdrosophila) <- paste(valid_terms, term_names, sep = "_")
    
    # Filter to gene universe if provided
    if (!is.null(gene_universe)) {
        GOdrosophila <- lapply(GOdrosophila, function(genes) {
            intersect(genes, gene_universe)
        })
        # Remove empty sets
        GOdrosophila <- GOdrosophila[sapply(GOdrosophila, length) > 0]
    }
    
    return(GOdrosophila)
}

# Create gene sets for your data
gene_universe <- rownames(sce)  # Use genes in your dataset
GOdrosophila <- create_drosophila_genesets(gene_universe = gene_universe)

message(paste("Created", length(GOdrosophila), "Drosophila GO gene sets"))

In [ ]:
# Example: Test cell types that appear in both datasets
# Replace with your actual cell types of interest
common_celltypes <- c("S25_13", "S25_7", "S25_9", "S24_13", "S24_7", "S24_9")  # Example cell types

# Filter for common cell types
sce_subset <- sce[, sce$cell_type %in% common_celltypes]

# Create cell type label matrix (binary matrix)
celltype_matrix <- model.matrix(~ sce_subset$cell_type - 1)
colnames(celltype_matrix) <- levels(as.factor(sce_subset$cell_type))

# Load example gene sets (replace with your own gene sets)
# You can use GO terms, pathway genes, or custom gene sets
# data(GOmouse)  # Example mouse GO terms - use GOhuman for human data
# gene_sets <- list(your_custom_set1 = c("GeneA", "GeneB", "GeneC"),
#                   your_custom_set2 = c("GeneX", "GeneY", "GeneZ"))

# Run supervised MetaNeighbor
AUROC_scores <- MetaNeighbor(
    dat = sce_subset,
    experiment_labels = as.numeric(factor(sce_subset$study_id)),
    celltype_labels = celltype_matrix,
    genesets = GOdrosophila,  # Replace with your gene sets
    bplot = TRUE,
    fast_version = TRUE
)

# Save results
write.csv(AUROC_scores, "AUROC_scores.csv")
print("AUROC scores for each gene set:")
print(head(AUROC_scores))